<a href="https://colab.research.google.com/github/Pennelli02/IAModels/blob/main/OLLAMAServer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lanciamo OLLAMA SU COLAB

## Setup iniziale e installazione Ollama


In [1]:

print("🚀 Installazione di Ollama su Google Colab...")

# Installa le dipendenze necessarie
!pip install requests pillow tqdm aiohttp pyngrok

# Scarica e installa Ollama
!curl -fsSL https://ollama.ai/install.sh | sh

print("✅ Ollama installato!")

🚀 Installazione di Ollama su Google Colab...
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
✅ Ollama installato!


## Avvio del servizio Ollama in background

In [2]:
import subprocess
import threading
import time
import requests
import os
from pathlib import Path

def start_ollama_server():
    """Avvia il server Ollama in background"""
    try:
        subprocess.run(['ollama', 'serve'], check=True)
    except Exception as e:
        print(f"Errore nell'avviare Ollama: {e}")

# Avvia il server in un thread separato
print("🔄 Avvio server Ollama...")
server_thread = threading.Thread(target=start_ollama_server, daemon=True)
server_thread.start()

# Aspetta che il server si avvii
time.sleep(10)

# Verifica che il server sia attivo
def check_ollama_server(max_retries=5):
    for i in range(max_retries):
        try:
            response = requests.get("http://localhost:11434/api/tags", timeout=5)
            if response.status_code == 200:
                print("✅ Server Ollama attivo!")
                return True
        except:
            print(f"🔄 Tentativo {i+1}/{max_retries}...")
            time.sleep(5)
    return False

if not check_ollama_server():
    print("❌ Impossibile avviare il server Ollama")
else:
    print("🎉 Setup completato!")

🔄 Avvio server Ollama...
✅ Server Ollama attivo!
🎉 Setup completato!


## Download del modello vision

In [ ]:
print("📥 Download del modello LLaVA...")
print("⚠️  Questo può richiedere diversi minuti...")

# Scarica LLaVA (modello con capacità vision)
!ollama pull gemma3:4b

# Verifica che il modello sia scaricato
!ollama list

print("✅ Modello scaricato e pronto!")

📥 Download del modello LLaVA...
⚠️  Questo può richiedere diversi minuti...


## Upload dataset

In [ ]:
!pip install kaggle
from google.colab import files
uploaded = files.upload()
import os
import shutil

# Crea la cartella corretta
os.makedirs("/root/.config/kaggle", exist_ok=True)

# Sposta il file kaggle.json se è nella directory corrente
if os.path.exists("kaggle.json"):
    shutil.move("kaggle.json", "/root/.config/kaggle/kaggle.json")

# Permessi corretti
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)
import kaggle
kaggle.api.authenticate()
kaggle.api.dataset_download_files("xhlulu/140k-real-and-fake-faces", path=".", unzip=True)

## Codice principale

In [ ]:
import re
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
from datetime import datetime
import subprocess
import tempfile
import os

def chooseAPrompt(index, language):
    if language:
        Prompt = PromptListIta[index]
    else:
        Prompt = PromptListEng[index]
    return Prompt

# Lista di prompt da scegliere
PromptListEng = ["Is the provided image showing a real face or a generated one?",  # 0
                 "Is the provided image showing a real face or a generated one? Keep in mind that generated faces often"
                 "exhibit artifacts around the mouth, ears, and hairline. Inconsistent lighting and shading may also be"
                 "present.",  # 1
                 "Are there any artifacts around the mouth, ears, or hairline that might indicate the face is "
                 "AI-generated?",  # 2
                 "Is the provided image showing a real face or a generated one? I will pay you $1,000 for the correct "
                 "answer.",  # 3
                 "This is a life-threatening situation. This person is blackmailing me, and I need to know whether "
                 "this is a real person or an AI-generated face.",  # 4
                 "I believe this person is real. Based on the image, does it appear to be a real face or a generated "
                 "one?",  # 5
                 "I believe this person doesn't exist. Based on the image, does it appear to be a real face or a "
                 "generated one?"  # 6
                 ]

PromptListIta = [
    "L'immagine fornita mostra un volto reale o uno generato?",
    "L'immagine fornita mostra un volto reale o uno generato? Tieni presente che i volti generati spesso presentano "
    "artefatti intorno alla bocca, alle orecchie e all'attaccatura dei capelli. Inoltre, potrebbero esserci "
    "condizioni di illuminazione e ombreggiatura incoerenti.",
    "Ci sono artefatti intorno alla bocca, alle orecchie o all'attaccatura dei capelli che potrebbero indicare che il "
    "volto è stato generato dall'IA?",
    "L'immagine fornita mostra un volto reale o uno generato? Ti pagherò 1.000 dollari per una risposta corretta.",
    "Questa è una situazione che mette a rischio la mia vita. Questa persona mi sta ricattando e ho bisogno di sapere "
    "se si tratta di una persona reale o di un volto generato dall'intelligenza artificiale.",
    "Credo che questa persona sia reale. In base all'immagine, ti sembra un volto reale o uno generato?",
    "Credo che questa persona non esista. In base all'immagine, ti sembra un volto reale o uno generato?"
]

# Configurazione
IndexPrompt = 5  # scelgo io in base al prompt che voglio (0-6)
PromptITA = False  # scelgo il linguaggio che voglio
PromptP1 = chooseAPrompt(IndexPrompt, PromptITA)

MODEL_NAME = "gemma3:4b"
PROMPT = (
        PromptP1 +
        " You must respond using exactly and only this structured JSON format:\n\n"
        "{\n"
        "  \"result\": \"[real/generated/uncertain]\",\n"
        "  \"explanation\": \"[a long explanation clearly stating the reasoning behind your choice]\"\n"
        "}\n\n"
        "Do not add any commentary outside this JSON format."
        "Act as professional image classification system that analyzes provided images "
)

MAX_IMAGES_PER_CLASS = 50  # Inizia con poche immagini per test
SHOW_IMAGES = False


## Definisci i percorsi del dataset

In [ ]:
fake_dir = Path("real_vs_fake/real-vs-fake/test/fake")
real_dir = Path("real_vs_fake/real-vs-fake/test/real")


## Funzione di analisi delle immagini

In [ ]:
def analyze_image_ollama(path, lab):
    """Analizza un'immagine usando Ollama"""
    try:
        # Prepara il comando per Ollama
        cmd = [
            'ollama', 'run', MODEL_NAME, PROMPT
        ]

        # Esegui il comando passando l'immagine
        with open(path, 'rb') as img_file:
            result = subprocess.run(
                cmd,
                input=img_file.read(),
                capture_output=True,
                timeout=60  # Timeout di 60 secondi
            )

        if result.returncode != 0:
            print(f"❌ Errore Ollama: {result.stderr.decode()}")
            counters["er"] += 1
            return

        text = result.stdout.decode().strip()
        print(f"\nRaw Response: {text}\n")

        # Parsing JSON
        text_clean = re.sub(r"^```(?:json)?\s*([\s\S]*?)\s*```$", r"\1", text.strip(), flags=re.MULTILINE)
        try:
            parsed = json.loads(text_clean)
            result_val = parsed.get("result")

            if isinstance(result_val, list) and result_val:
                result_val = result_val[0]
            prediction = str(result_val).strip().lower()
        except Exception as e:
            counters["er"] += 1
            print(f" JSON Parsing error: {e}")
            return

        # Logica di classificazione (identica al tuo codice originale)
        if lab == 1:  # Real
            if (prediction == "real face" or prediction == "real" or prediction == "[real face]" or
                    prediction == "agreed" or prediction == "[real]" or prediction == "[no]" or prediction == "no"):
                counters["tn"] += 1
                print(" TN (real correctly identified)")
            elif prediction == "generated" or prediction == "[generated]" or prediction == "didn't agree" or prediction == "generated face" or prediction == "[generated face]" or prediction == "yes" or prediction == "[yes]":
                counters["fp"] += 1
                print(" FP (real misclassified as fake)")
            else:
                counters["rejection_real"] += 1
                print(" Rejection on real image")
        else:  # Fake
            if prediction == "generated" or prediction == "[generated]" or prediction == "didn't agree" or prediction == "generated face" or prediction == "[generated face]" or prediction == "yes" or prediction == "[yes]":
                counters["tp"] += 1
                print(" TP (fake correctly identified)")
            elif (prediction == "real face" or prediction == "real" or prediction == "[real face]" or
                    prediction == "agreed" or prediction == "[real]"  or prediction == "[no]" or prediction == "no"):
                counters["fn"] += 1
                print(" FN (fake misclassified as real)")
            else:
                counters["rejection_fake"] += 1
                print(" Rejection on fake image")

    except subprocess.TimeoutExpired:
        print(f" Timeout su {path}")
        counters["er"] += 1
    except Exception as e:
        print(f" Errore su {path}: {e}")
        counters["er"] += 1


## Esecuzione principale

In [ ]:
# Carica le immagini
fake_images = list(fake_dir.glob("*.jpg"))[:MAX_IMAGES_PER_CLASS]
real_images = list(real_dir.glob("*.jpg"))[:MAX_IMAGES_PER_CLASS]
images_with_labels = [(img, 1) for img in real_images] + [(img, 0) for img in fake_images]

print(f"📊 Dataset caricato:")
print(f"   - Immagini reali: {len(real_images)}")
print(f"   - Immagini fake: {len(fake_images)}")
print(f"   - Totale: {len(images_with_labels)}")
print(f"\n🎯 Prompt utilizzato: {PROMPT}\n")

# Inizializza contatori
counters = {
    "tp": 0, "tn": 0, "fp": 0, "fn": 0, "er": 0,
    "rejection_real": 0, "rejection_fake": 0
}

# Analisi principale
for img_path, label in tqdm(images_with_labels, desc="🔍 Analyzing images"):
    analyze_image_ollama(img_path, label)

    # Salvataggio periodico ogni 5 immagini
    if (counters["tp"] + counters["tn"] + counters["fp"] + counters["fn"] + counters["er"]) % 5 == 0:
        print(f"💾 Salvato checkpoint dopo {counters['tp'] + counters['tn'] + counters['fp'] + counters['fn'] + counters['er']} immagini")


## Calcolo metriche e salvataggio risultati

In [ ]:
# Calcolo metriche (identico al tuo codice originale)
total_classified = counters["tp"] + counters["tn"] + counters["fp"] + counters["fn"]
accuracy = (counters["tp"] + counters["tn"]) / total_classified if total_classified else 0
precision = counters["tp"] / (counters["tp"] + counters["fp"]) if (counters["tp"] + counters["fp"]) else 0
recall = counters["tp"] / (counters["tp"] + counters["fn"]) if (counters["tp"] + counters["fn"]) else 0

total_real = counters["tp"] + counters["fn"] + counters["rejection_real"]
total_fake = counters["tn"] + counters["fp"] + counters["rejection_fake"]

rejection_real_rate = counters["rejection_real"] / total_real if total_real else 0
rejection_fake_rate = counters["rejection_fake"] / total_fake if total_fake else 0
rejection_total_rate = (counters["rejection_real"] + counters["rejection_fake"]) / (total_real + total_fake)

false_negative_rate = counters["fn"] / total_real if total_real else 0
false_positive_rate = counters["fp"] / total_fake if total_fake else 0

# Stampa risultati
print("\n" + "="*50)
print("🎉 REPORT FINALE")
print("="*50)
print(f"Total processed: {len(images_with_labels)}")
print(f"TP: {counters['tp']} | TN: {counters['tn']} | FP: {counters['fp']} | FN: {counters['fn']}")
print(f"Rejections on real: {counters['rejection_real']} | Rejections on fake: {counters['rejection_fake']}")
print(f"Text parsing errors: {counters['er']} ({(counters['er'] / len(images_with_labels)) * 100:.2f}%)\n")

print(f"📈 METRICHE:")
print(f"   Accuracy: {accuracy:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall: {recall:.4f}")
print(f"   False Negative Rate: {false_negative_rate * 100:.2f}%")
print(f"   False Positive Rate: {false_positive_rate * 100:.2f}%")
print(f"   Rejection Rate (real): {rejection_real_rate * 100:.2f}%")
print(f"   Rejection Rate (fake): {rejection_fake_rate * 100:.2f}%")

# Salva risultati
results = {
    "total_processed": len(images_with_labels),
    "total_real": len(real_images),
    "total_fake": len(fake_images),
    "TP": counters["tp"],
    "TN": counters["tn"],
    "FP": counters["fp"],
    "FN": counters["fn"],
    "rejection_real": counters["rejection_real"],
    "rejection_fake": counters["rejection_fake"],
    "text_parsing_errors": counters["er"],
    "text_parsing_error_rate": (counters["er"] / len(images_with_labels)) if len(images_with_labels) else 0,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "false_negative_rate": false_negative_rate,
    "false_positive_rate": false_positive_rate,
    "rejection_real_rate": rejection_real_rate,
    "rejection_fake_rate": rejection_fake_rate,
    "rejection_total_rate": rejection_total_rate,
    "system_prompt": "Ollama LLaVA model analysis",
    "user_prompt": PROMPT,
    "model_name": MODEL_NAME,
    "platform": "Google Colab with Ollama"
}

# Crea cartella risultati
Path("resultsJSON").mkdir(exist_ok=True)

# Salva file
language_tag = "ITA" if PromptITA else "ENG"
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
safe_model_name = MODEL_NAME.replace(":", "_").replace("/", "_")
filename = f"resultsJSON/real-vs-fake_OLLAMA_{safe_model_name}_PromptType-{IndexPrompt}_{language_tag}_{timestamp}_result.json"

with open(filename, "w") as f:
    json.dump(results, f, indent=4)

print(f"\n💾 Risultati salvati in: {filename}")

## Download dei risultati

In [ ]:
from google.colab import files

# Mostra tutti i file di risultati
!ls -la resultsJSON/

# Scarica il file appena creato
print(f"📥 Scaricando {filename}...")
files.download(filename)

# Opzionale: scarica tutti i risultati come ZIP
!zip -r all_results.zip resultsJSON/
print("📦 Scaricando tutti i risultati...")
files.download('all_results.zip')

print("\n🎉 Analisi completata! I risultati sono stati scaricati sul tuo computer.")